James Caldwell <br>
October 2025 <br>

This script automates prize emails for IGOW/RaceGOW

In [ ]:
import numpy as np
import pandas as pd
import tkinter as tk
from tkinter import ttk
import pandas as pd
import time
from datetime import datetime
import re
# from __future__ import print_function
import os.path
import base64
from email.mime.text import MIMEText
from google.auth.transport.requests import Request
from google.oauth2.credentials import Credentials
from google_auth_oauthlib.flow import InstalledAppFlow
from googleapiclient.discovery import build

In [2]:
global competition_name 
competition_name = "RaceGOW5"

In [3]:
def url_to_dataframe(sheet_url,skiprows):    
    try:
        pattern = r"https://docs\.google\.com/spreadsheets/d/([a-zA-Z0-9_-]+)/edit\?gid=(\d+)"
        match = re.search(pattern, sheet_url)
        sheet_id, gid = match.groups()
        csv_url = f"https://docs.google.com/spreadsheets/d/{sheet_id}/export?format=csv&gid={gid}"
        df = pd.read_csv(csv_url,skiprows=skiprows)
        # print("Sheet ID:", sheet_id)
        # print("GID:", gid)
    except Exception as e:
        print("Error accessing URL:", e)
    return df

In [4]:
links_df = pd.read_csv('links.txt',sep='\t')

all_dfs = {}

# Iterate and load data from all the sheets in the .txt file
for index, row in links_df.iterrows():
    sheet_url = row['Value']
    sheet_name = row['Variable']
    if index == 0:
        skiprows = 0
    else:
        skiprows = 1
    df = url_to_dataframe(sheet_url,skiprows) 

    df.iloc[:, 0] = df.iloc[:, 0].replace("Missing value", pd.NA).ffill() # fill down 1st column with missing values for Track 1, Track 2, etc.

    all_dfs[sheet_name] = df         

In [5]:
registration_df = all_dfs['Registration Sheet link']
prize_dict = {k: v for k, v in all_dfs.items() if k != 'Registration Sheet link'}

In [6]:
prize_groups = list(prize_dict.keys())
# print(prize_groups) # ['Prizes for Everyone', 'Prizes for Bonus Tier', 'Emax Bonus Prizes']

In [7]:
registration_email_column_name = "Email" # Column name for email from registration sheet
paypal_email_column_name = "PayPal Email" # Column name for paypal email from registration sheet
registration_df['Email'] = 'charlottesville.drone@gmail.com'
registration_df['PayPal Email'] = 'charlottesville.drone@gmail.com'
callsign_and_email = registration_df[['What is your Pilot Callsign (Handle)?',registration_email_column_name,paypal_email_column_name]]

In [8]:
def set_week_dropdown_list(selected_prize_df):
    # Generate list of weeks for GUI dropdown
    global week_dropdown_list 
    week_dropdown_list = sorted(selected_prize_df.iloc[:, 0].unique()) # Need to change open_prizes to prize group

# Get's the selected week from the prize list and adds the email from registration
def get_winner_info(prize_group,week):

    selected_prize_df = prize_dict[prize_group]
    set_week_dropdown_list(selected_prize_df)
    
    # Filter prizes for the week
    week_df = selected_prize_df[selected_prize_df.iloc[:, 0] == week].copy()
    
    # Merge with callsign_and_email to get both Email and PayPal Email
    # Assuming callsign_and_email has columns: [callsign, Email, PayPal Email]
    week_df = week_df.merge(
        callsign_and_email,
        left_on=week_df.columns[1],  # winner column in week_df
        right_on=callsign_and_email.columns[0],  # callsign column
        how='left'
    )
    
    # Optional: drop the extra 'key' column from merge if needed
    week_df.drop(columns=[callsign_and_email.columns[0]], inplace=True)
    
    return week_df

# test usage
week = 'Training Ground #0'
winner_info = get_winner_info('Prizes for Everyone',week)
winner_info

,Challenge,Winner,Prize Sponsor,Prize Detail,~Retail Value,Total(s),Unnamed: 6,Preseason,Regular Season,Email,PayPal Email
0,Training Ground #0,自由落体_FREEFALLALL,Velocidrone,$50 CASH via Paypal,$50.00,$258.00,NaN,"$2,680.00","$11,103.00",charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
1,Training Ground #0,VelocityX,Pyrodrone,$50 OFF code at Pyrodrone.com,$50.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
2,Training Ground #0,Tarasik,VIFLY + Skittles,Whoopstor V3 1S Charger + 3D Printer Charger/W...,$48.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
3,Training Ground #0,c1nemat1ck,Average Jane and Joes,$25 CASH via Paypal,$25.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
4,Training Ground #0,MadBat,Crazy_Like_A,$20 CASH via Paypal,$20.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
5,Training Ground #0,Dudemanshu,weBLEEDfpv,$20 OFF code at weBLEEDfpv.com,$20.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
6,Training Ground #0,RetryWin,TweetFPV,$15 OFF code to Tweetfpv.com,$15.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
7,Training Ground #0,westexfpv,Decks of Dexterity,Free Steam Copy of this Game https://www.decks...,$15.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com
8,Training Ground #0,L2abbit,Nick Burns,$15 Are you here? Must be present in livestrea...,$15.00,NaN,NaN,NaN,NaN,charlottesville.drone@gmail.com,charlottesville.drone@gmail.com


https://developers.google.com/workspace/docs/api/quickstart/python#enable_the_api

https://console.cloud.google.com/projectselector2/auth/overview?supportedpurview=project&authuser=2

Make new project (i called mine IGOW)

Enable gmail api:  click "enable api"

Desktop client 1

download json file and save to working directory. rename to credentials.json

add self to tester list under "audience"


For later:
If authenication expires (every 2-3 weeks?)
Go to:
https://console.cloud.google.com/auth/clients?authuser=0&project=igow-476417&supportedpurview=project
Clients tab
Creat new desktop client, download new .json file, delete old credentials.json and delete the token.json as well.


In [9]:
# This section sets up the Gmail api settings
    # This section has no gui associated with it

# If modifying scopes, delete the file token.json.
SCOPES = ['https://www.googleapis.com/auth/gmail.send']

def gmail_service_setup():
    """Authenticate and return a Gmail API service."""
    creds = None
    # token.json stores user’s access/refresh tokens after first login
    if os.path.exists('token.json'):
        creds = Credentials.from_authorized_user_file('token.json', SCOPES)
    # If no valid credentials, prompt user login
    if not creds or not creds.valid:
        if creds and creds.expired and creds.refresh_token:
            creds.refresh(Request())
        else:
            flow = InstalledAppFlow.from_client_secrets_file(
                'credentials.json', SCOPES)
            creds = flow.run_local_server(port=0)
        # Save the credentials for next time
        with open('token.json', 'w') as token:
            token.write(creds.to_json())
    return build('gmail', 'v1', credentials=creds)

def create_message(sender, to, subject, message_text):
    """Create a MIMEText email and encode in base64."""
    message = MIMEText(message_text)
    message['to'] = to
    message['from'] = sender
    message['subject'] = subject
    raw = base64.urlsafe_b64encode(message.as_bytes())
    return {'raw': raw.decode()}

def send_message(service, user_id, message):
    """Send an email via Gmail API."""
    sent_message = service.users().messages().send(userId=user_id, body=message).execute()
    # print(f"Message Id: {sent_message['id']}") # message id, uncomment for troubleshooting
    return sent_message

In [10]:
def log_email_sent(message): 
    with open("email_log.txt", "a") as log_file: 
        log_file.write(message + "\n")


def safe_str(x):
    """Convert any value to a safe string (replace NaN with empty)."""
    try:
        if pd.isna(x):
            return ""
    except TypeError:
        pass
    return str(x)

In [11]:
# Create main window
root = tk.Tk()
root.state("zoomed")
root.title("IGOW/RaceGOW Prize Automation App")

# Instruction Label
instruction_label = tk.Label(
    root,
    text="1. Select checkbox for testing vs sending actual emails \n2. Select a track week/prize group from the drop downs \n3. Click generate and confirm button \n4. Remove any emails you don't want to send \n5. Click 'Send Emails' to send winner emails.",
    wraplength=350,     # wrap text after 350 pixels
    justify="left",     # align text to the left
    # fg="gray"           # grey color for subtle look
)
instruction_label.pack(pady=(5, 10))

# # ===== TESTING CHECKBOX + DROPDOWN SECTION =====
tk.Label(
    root,
    text="------------ TESTING ---------- \nLeave unchecked to send all emails to email below. "
         "Check to send actual prize emails."
).pack(pady=5)

test_checkbox_var = tk.IntVar(value=0)
checkbox = tk.Checkbutton(
    root, 
    variable=test_checkbox_var,
    font=("Arial", 16)  # Makes the checkbox bigger
)
checkbox.pack(pady=5)

# --- Testing dropdown ---
tk.Label(root, text="TESTING email: all emails will be sent here instead of actual winner emails (if box above is unchecked)").pack(pady=5)
test_email_var = tk.StringVar(value='charlottesville.drone@gmail.com')  # default value
test_dropdown = ttk.Combobox(
    root,
    textvariable=test_email_var,
    values=['charlottesville.drone@gmail.com', 'igowhoop@gmail.com'],
    state="readonly"
)
test_dropdown.pack(pady=5)

# ===== Prize Group SELECTION DROPDOWN =====
tk.Label(root, text="------------------------------------------------------------------------------------------------------------------------------------------------\n\nSelect Prize Group:").pack(pady=5)
prize_group_var = tk.StringVar()
prize_group_dropdown = ttk.Combobox(
    root,
    textvariable=prize_group_var,
    values=prize_groups,
    state="readonly"
)
prize_group_dropdown.pack(pady=5)
prize_group_dropdown.current(0)  # sets the default selection

# ===== TRACK SELECTION DROPDOWN =====
tk.Label(root, text="Select track #:").pack(pady=5)
# week_dropdown_list = ["Week 1", "Week 2", "Week 3"]  # placeholder list
track_week_var = tk.StringVar()
track_dropdown = ttk.Combobox(
    root,
    textvariable=track_week_var,
    values=week_dropdown_list, # Can we assume that the week list is the same for all prize groups? Right now, that is assumed.
    state="readonly"
)
track_dropdown.pack(pady=5)
# track_dropdown.current(2)  # sets the default selection

# ===== Output Label =====
output_label = tk.Label(root, text="", fg="blue")
output_label.pack(pady=5)

# ===== Button =====
def generate_table():
    # Clear previous rows
    for row in tree.get_children():
        tree.delete(row)
    
    prize_group = prize_group_var.get()
    selected_week = track_week_var.get()
    winner_info = get_winner_info(prize_group,selected_week)
    global winner_abbreviated
    winner_abbreviated = winner_info[['Winner','Prize Detail','Prize Sponsor','Email','PayPal Email']]
    
    # Insert rows into Treeview
    for _, row in winner_abbreviated.iterrows():
        tree.insert("", tk.END, values=list(row))
    
    output_label.config(text=f"Generating emails for {selected_week} winners:")

# ===== Button =====
submit_button = tk.Button(root, text="Generate prize table and emails to confirm before sending", command=generate_table)
submit_button.pack(pady=10)

# ===== Treeview for Table =====
columns = ("Winner", "Prize Detail","Prize Sponsor", "Email", "PayPal Email")
tree = ttk.Treeview(root, columns=columns, show="headings", height=10)
for col in columns:
    tree.heading(col, text=col)
    tree.column(col, width=200)  # adjust width as needed
tree.pack(pady=5, fill=tk.X)

# Add vertical scrollbar
scrollbar = ttk.Scrollbar(root, orient="vertical", command=tree.yview)
tree.configure(yscrollcommand=scrollbar.set)
scrollbar.pack(side=tk.RIGHT, fill=tk.Y)

# ===== Output Label =====
output_label2 = tk.Label(root, text="", fg="blue")
output_label2.pack(pady=5)

# This removes pilots from the email list, presumably if there's an error in the data for some reason.
def error_values_check():
    global winner_abbreviated
    input_text = error_var.get().strip()
    if input_text:
        skip_list = [item.strip() for item in input_text.split(',')]
    else:
        skip_list = []
    # Keep only rows where 'pilot name' is NOT in skip_list
    winner_abbreviated = winner_abbreviated[~winner_abbreviated['Winner'].isin(skip_list)]

# ===== Button =====
def on_submit():

    output_label2.config(text=f"Sending emails...")
    
    root.after(500, send_email) # wait 500ms before calling send_email to allow label update

def send_email():
    
    # Start log with date and time:
    timestamp = datetime.now().strftime("%Y-%m-%d %H:%M")
    log_email_sent('----'+timestamp+'----')

    error_values_check() # Removes rows manually if desired for errors

    try:
        service = gmail_service_setup()

        for idx, row in winner_abbreviated.iterrows():
            if test_checkbox_var.get():
                to_email = row['Email']
            else:
                to_email = test_email_var.get()

            to_email = safe_str(to_email)

            winner = row['Winner']
            prize = row['Prize Detail']
            prize_sponsor = row['Prize Sponsor']
            paypal_email = row['PayPal Email']

            if 'cash' in prize.lower(): # Paypal cash prize
                prize_or_paypal_message = f'Please reply and confirm your PayPal email and I will send the prize: {paypal_email}'
            else:
                prize_or_paypal_message = 'Please reply with your shipping address and information for your prize to be sent to you.'
            # print(prize_or_paypal_message)

            try:
                email_msg = create_message(
                    sender="charlottesville.drone@gmail.com",
                    to=to_email,
                    subject=f"Claim Your {competition_name} Prize!",
                message_text = (
                    f"Congratulations {winner}!\n\n"
                    f"You won a '{prize}' in {competition_name} sponsored by: {prize_sponsor}\n\n"
                    f"{prize_or_paypal_message}\n\n"
                    f"Thanks for participating in {competition_name} and good luck in the rest of the game!"
                    + (f"\n\nTESTING condition, email would have gone to: {row['Email']}" if not test_checkbox_var.get() else "" )
                )
                    )
                send_message(service, "me", email_msg)
                log_message = str(idx) + '. Email sent to ' + winner + ' / ' + to_email + ' / ' + prize
            except Exception as err:
                print(err)
                log_message = (
                    str(idx) +
                    ". Failed to send email to " + str(winner)
                    + " with email: " + str(to_email)
                    + ": " + str(err)
                    + " prize: " + str(prize)
                )
            log_email_sent(log_message)
            
            if test_checkbox_var.get():
                output_label2.config(text=f"Sending emails...{winner}")
            else:
                output_label2.config(text=f"Sending to testing email {to_email} for winner {winner}")
            time.sleep(0.25)
            root.update()
                   
        time.sleep(0.25)
        output_label2.config(text=f"Emails sent!")
    except Exception as e: # we'll end up here if the google email api setup didn't work
        output_label2.config(text=f"Incorrect/expired email api settings or invalid email parameters, reason: {e}")

# ===== Skip emails for error =====
tk.Label(root, text="If any rows above have errors, enter the pilot ID(s) separated by a comma below. Program will skip their emails and you can email manually.\nIf table above is correct, leave blank").pack(pady=5)
error_var = tk.StringVar()
error_entry = tk.Entry(
    root, 
    textvariable=error_var,
    width=30  # adjust width as needed
)
error_entry.pack(pady=5)

# ===== Button =====
submit_button = tk.Button(root, text="Send emails", command=on_submit)
submit_button.pack(pady=10)

# ===== Close handler =====
def on_close():
    root.quit()
    root.destroy()

root.protocol("WM_DELETE_WINDOW", on_close)
root.mainloop()
